# PhoBERT NER (SKILL / TIME / LOC) — bản chia sẻ cho team

**Dùng cho người KHÔNG cần hiểu code, chỉ cần chạy để dùng model đã train sẵn.**

Trước khi chạy:
1. `Runtime → Change runtime type → T4 GPU` (hoặc GPU nào có sẵn).
2. Có file `phobert_ner.zip` (chủ project gửi cho bạn) — chứa code, không chứa model.
3. Model đã train (`outputs/`) được lưu trên Google Drive chung của team. Chủ project phải **share** thư mục đó
   cho bạn (hoặc bạn bấm "Add shortcut to Drive" để nó xuất hiện trong `My Drive` của bạn).
   Hỏi chủ project đường dẫn chính xác (tên thư mục) nếu bước 5 bên dưới báo lỗi không tìm thấy.

Chạy từng cell **theo đúng thứ tự, từ trên xuống**, không nhảy cóc. Nếu gặp lỗi, xem mục **Xử lý lỗi thường gặp** ở cuối.

In [ ]:
#@title Bước 0: Kiểm tra GPU
!nvidia-smi
import torch; print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())

## Bước 1: Đưa code lên Colab

In [ ]:
#@title Bước 1a: Upload phobert_ner.zip
# Xoá zip cũ (nếu chạy lại nhiều lần, tránh Colab lưu trùng tên thành "(1).zip")
!rm -f /content/*.zip
from google.colab import files
up = files.upload()  # chọn đúng file phobert_ner.zip, khoảng 10MB

In [ ]:
#@title Bước 1b: Giải nén + kiểm tra
!ls -la /content/*.zip
!unzip -q -o /content/phobert_ner.zip -d /content/
%cd /content
!ls

Sau bước 1b phải thấy `VPhoBertTagger`, `data_dir`, `data_build`, `train_*.sh` trong danh sách `!ls`.
Nếu không thấy — đừng chạy tiếp, xem mục xử lý lỗi "Upload/giải nén không ra code" ở cuối.

## Bước 2: Cài môi trường (Java + Python packages)

In [ ]:
#@title Bước 2a: Java (VnCoreNLP cần Java để tách từ)
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version

In [ ]:
#@title Bước 2b: Thư viện Python (giữ nguyên torch CUDA có sẵn của Colab)
!grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt
import pandas, subprocess
if int(pandas.__version__.split('.')[0]) >= 3:
    subprocess.run(['pip', '-q', 'install', 'pandas>=2.2,<3'], check=True)
    print('Đã hạ pandas xuống <3 -> Runtime > Restart session rồi chạy LẠI TỪ BƯỚC 1b (vì restart session sẽ về /content gốc, không mất file đã giải nén)')

In [ ]:
#@title Bước 2c: Tải VnCoreNLP (dùng để tách từ tiếng Việt)
import os
VN = os.path.abspath('vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
!find {VN} -type f | xargs ls -la

## Bước 3: Lấy model đã train sẵn (KHÔNG cần train lại)

Model (`best_model.pt`) đã được train 1 lần và lưu trên Google Drive chung của team.
Bạn chỉ cần copy về, không cần tốn thời gian train lại — trừ khi muốn tự thử nghiệm (xem Bước 3b, không bắt buộc).

In [ ]:
#@title Bước 3a: Mount Drive + copy model đã train
from google.colab import drive
drive.mount('/content/drive')

# SỬA đường dẫn này theo đúng tên thư mục chủ project đã share cho bạn.
# Nếu chủ project cho bạn "Add shortcut to Drive", nó sẽ nằm trong MyDrive của chính bạn.
SHARED_OUTPUTS = "/content/drive/MyDrive/phobert_ner_outputs/outputs"

!cp -r "{SHARED_OUTPUTS}" /content/outputs
!ls outputs/*/best_model.pt

Nếu lệnh `!ls outputs/*/best_model.pt` báo lỗi "No such file or directory" → đường dẫn `SHARED_OUTPUTS` sai hoặc bạn chưa được share quyền truy cập.
Hỏi lại chủ project tên thư mục đúng, hoặc nhờ họ "Share" / gửi bạn link "Add shortcut to Drive".

### Bước 3b (KHÔNG bắt buộc): Tự train lại từ đầu
Chỉ làm nếu bạn thực sự cần train lại (đổi dữ liệu, đổi tham số...). Mỗi lệnh ~20-40 phút trên GPU T4.
Bỏ dấu `#` ở dòng bạn muốn chạy. Không chạy 2 dòng song song (dùng chung cache trong `data_dir/`).

In [ ]:
#@title Bước 3b: (tuỳ chọn) Train lại
# !bash train_softmax.sh 2>&1 | grep -v 'it/s]'
# !bash train_crf.sh 2>&1 | grep -v 'it/s]'
# !bash train_lstm_crf.sh 2>&1 | grep -v 'it/s]'

# Sau khi train xong, LƯU NGAY vào Drive chung trước khi làm gì khác (tránh mất khi Colab bị ngắt):
# !mkdir -p "/content/drive/MyDrive/phobert_ner_outputs" && cp -r outputs logs "/content/drive/MyDrive/phobert_ner_outputs/"
print('Bỏ dấu # ở các dòng trên nếu muốn train lại.')

## Bước 4: Thử model với câu của bạn

In [ ]:
#@title Bước 4: Nhập câu và xem model đoán SKILL/TIME/LOC
CAU_CAN_THU = "tôi giỏi tiếng anh, rảnh vào tối và ở Hải Châu"  #@param {type:"string"}
ARCH = "softmax"  #@param ["softmax", "crf", "lstm_crf"]

import subprocess
cmd = f'printf "{CAU_CAN_THU}\\n" | python VPhoBertTagger/main.py predict --model_path outputs/{ARCH}/best_model.pt'
out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
for line in out.stdout.splitlines():
    if 'EOFError' not in line:
        print(line)

---
## Xử lý lỗi thường gặp

Chạy cell "Cứu hộ nhanh" ngay dưới nếu gặp **bất kỳ** lỗi nào trong danh sách sau — nó làm lại toàn bộ Bước 2+3 một lượt:

- `ModuleNotFoundError: No module named 'torchcrf'` / `'jsonlines'` / bất kỳ module nào khác
  → thiếu thư viện Python, do Colab bị ngắt kết nối (reset runtime) làm mất hết package đã cài.
- `FileNotFoundError: File "vncorenlp/VnCoreNLP-1.1.1.jar" was not found`
  → thư mục `vncorenlp/` bị mất do reset runtime, hoặc đang chạy lệnh từ sai thư mục (phải chạy từ `/content`).
- `FileNotFoundError: ... 'outputs/<arch>/best_model.pt'`
  → model đã train bị mất do reset runtime. Chạy lại Bước 3a (copy từ Drive), KHÔNG cần train lại nếu đã có trên Drive.
- `python3: can't open file '/content/VPhoBertTagger/main.py': No such file or directory`
  → code chưa được giải nén trong phiên làm việc hiện tại. Chạy lại Bước 1 (upload + giải nén).
- `unzip: cannot find or open /content/phobert_ner.zip...`
  → chưa upload file, hoặc Colab lưu file thành tên khác (`phobert_ner (1).zip`...) do bạn upload nhiều lần.
  Chạy `!ls -la /content/*.zip` để xem tên thật, rồi `!unzip -q -o "/content/<tên đúng>.zip" -d /content/`.

**Nguyên nhân gốc của gần hết các lỗi trên: Colab (bản free) tự ngắt/reset runtime khi để không lâu.**
Mỗi lần reset, `/content/` bị xoá sạch — phải chạy lại từ Bước 1 đến Bước 3 (không cần train lại nếu model đã có trên Drive).

In [ ]:
#@title CỨU HỘ NHANH: chạy lại Bước 2 + 3 một lượt (không train lại)
import os

assert os.path.isdir('/content/VPhoBertTagger'), 'Chưa có code — chạy lại Bước 1 (upload + giải nén) trước.'

get_ipython().system("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null")
get_ipython().system("grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt")
get_ipython().system("pip -q install -r /tmp/req_colab.txt")

VN = os.path.abspath('vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    get_ipython().system(f"wget -q -nc -O {VN}/{f} {B}/{f}")

from google.colab import drive
drive.mount('/content/drive')
SHARED_OUTPUTS = "/content/drive/MyDrive/phobert_ner_outputs/outputs"  # sửa nếu cần
get_ipython().system(f'cp -r "{SHARED_OUTPUTS}" /content/outputs')

print('Xong. Thử lại Bước 4.')